# FIFA World Cup 2026 — Tournament Prediction

End-to-end pipeline that:

1. **Trains** Random Forest models on historical World Cup match scores (2002–2022, with Kaggle pre-tournament features)
2. **Predicts** every 2026 group-stage fixture from `Data/clean_fifa_worldcup_fixture.csv`
3. **Builds** group standings (points + goal difference)
4. **Simulates** a simplified knockout bracket through to a predicted champion

**Outputs:** `Data/fifa_worldcup_2026_predictions.csv`, `Data/fifa_worldcup_2026_standings.csv`, `Data/predicted_tournament_results.csv`, `Data/final_tournament_standings.csv`

> **Note:** Qualification uses the top 16 teams overall (not FIFA's real group-winner + best-third rules). Knockout pairings follow standings order, not the official draw.

In [1]:
# Core libraries for data handling and score-prediction models
import sys

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error

sys.path.insert(0, "scripts")
from kaggle_team_features import (
    default_kaggle_dir,
    enrich_matches,
    match_model_feature_columns,
    training_matches,
)

/Users/nigelcuschieri/Development/world-cup-predictor/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Step 1 — Train score-prediction models

Load `Data/clean_fifa_worldcup_matches.csv` (Wikipedia scores + Kaggle features from `scripts/scrape_historical_matches.py`). Train on **2002+** rows with FIFA rank/points, 4-year form, host flags, squad market value (median-imputed when missing), squad age, World Cup titles, same-confederation matchups, and home-minus-away diffs.

In [2]:
# Historical World Cup results (re-run scrape script if Kaggle columns are missing)
df = pd.read_csv("Data/clean_fifa_worldcup_matches.csv").dropna(
    subset=["HomeGoals", "AwayGoals"]
)
if "SameContinent" not in df.columns:
    df = enrich_matches(df, default_kaggle_dir())

df_train = training_matches(df)

team_encoder = LabelEncoder()
team_encoder.fit(pd.concat([df_train["HomeTeam"], df_train["AwayTeam"]]).unique())
df_train = df_train.copy()
df_train["HomeTeamEncoded"] = team_encoder.transform(df_train["HomeTeam"])
df_train["AwayTeamEncoded"] = team_encoder.transform(df_train["AwayTeam"])

FEATURE_COLS = match_model_feature_columns()
X = df_train[FEATURE_COLS]
feature_imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(
    feature_imputer.fit_transform(X), columns=FEATURE_COLS, index=df_train.index
)

y_home = df_train["HomeGoals"]
y_away = df_train["AwayGoals"]

# 80/20 split for optional evaluation (models train on X_train below)
X_train, X_test, y_train_home, y_test_home = train_test_split(
    X, y_home, test_size=0.2, random_state=42
)
_, _, y_train_away, y_test_away = train_test_split(
    X, y_away, test_size=0.2, random_state=42
)

In [3]:
# Separate models let home/away scoring patterns differ
home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)

home_model.fit(X_train, y_train_home)
away_model.fit(X_train, y_train_away)


RandomForestRegressor(random_state=42)

## Step 2 — Predict 2026 group-stage fixtures

Apply the trained models to every scheduled 2026 match and save predicted scores.

In [4]:
fixtures = pd.read_csv("Data/clean_fifa_worldcup_fixture.csv")
fixtures.rename(columns={"home": "HomeTeam", "away": "AwayTeam", "year": "Year"}, inplace=True)

fixtures = enrich_matches(fixtures, default_kaggle_dir())
fixtures["HomeTeamEncoded"] = fixtures["HomeTeam"].apply(
    lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1
)
fixtures["AwayTeamEncoded"] = fixtures["AwayTeam"].apply(
    lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1
)

X_fixtures = pd.DataFrame(
    feature_imputer.transform(fixtures[FEATURE_COLS]),
    columns=FEATURE_COLS,
    index=fixtures.index,
)

# Round continuous predictions to whole goals for standings math
fixtures["PredictedHomeGoals"] = home_model.predict(X_fixtures).round().astype(int)
fixtures["PredictedAwayGoals"] = away_model.predict(X_fixtures).round().astype(int)

fixtures['Winner'] = np.select(
    [
        fixtures['PredictedHomeGoals'] > fixtures['PredictedAwayGoals'],
        fixtures['PredictedHomeGoals'] < fixtures['PredictedAwayGoals'],
    ],
    [fixtures['HomeTeam'], fixtures['AwayTeam']],
    default='Draw',
)


In [5]:
# One row per fixture with predicted scores and winner
fixtures.to_csv("Data/fifa_worldcup_2026_predictions.csv", index=False)


## Step 3 — Compute group standings

Aggregate predicted results into points (3 win / 1 draw) and goal difference, then rank teams.

In [6]:
group_stage_results = pd.read_csv("Data/fifa_worldcup_2026_predictions.csv")
print(group_stage_results.head())

standings = {}  # team -> {Points, GD}

for _, row in group_stage_results.iterrows():
    home, away = row['HomeTeam'], row['AwayTeam']
    home_goals = int(row['PredictedHomeGoals'])
    away_goals = int(row['PredictedAwayGoals'])
    standings.setdefault(home, {'Points': 0, 'GD': 0})
    standings.setdefault(away, {'Points': 0, 'GD': 0})

    standings[home]['GD'] += home_goals - away_goals
    standings[away]['GD'] += away_goals - home_goals

    if home_goals > away_goals:
        standings[home]['Points'] += 3
    elif away_goals > home_goals:
        standings[away]['Points'] += 3
    else:
        standings[home]['Points'] += 1
        standings[away]['Points'] += 1

standings_df = pd.DataFrame.from_dict(standings, orient='index').reset_index().rename(columns={'index': 'Team'})
standings_df = standings_df.sort_values(by=['Points', 'GD'], ascending=False)


         HomeTeam     score        AwayTeam  Year  HomeFifaRank  \
0          Mexico   Match 1    South Africa  2026          15.0   
1     South Korea   Match 2  Czech Republic  2026          25.0   
2  Czech Republic  Match 25    South Africa  2026          41.0   
3          Mexico  Match 28     South Korea  2026          15.0   
4  Czech Republic  Match 53          Mexico  2026          41.0   

   HomeFifaPoints  HomeGoalsScoredLast4Y  HomeGoalsReceivedLast4Y  \
0         1681.03                   80.0                     52.0   
1         1588.66                   76.0                     36.0   
2         1501.38                   66.0                     38.0   
3         1681.03                   80.0                     52.0   
4         1501.38                   66.0                     38.0   

   HomeWinsLast4Y  HomeLossesLast4Y  ...  DiffDrawsLast4Y  \
0            27.0              12.0  ...             -5.0   
1            21.0               6.0  ...              1.0   

In [7]:
# Ranked table of all teams after the simulated group stage
standings_df.to_csv("Data/fifa_worldcup_2026_standings.csv", index=False)


## Step 4 — Simulate knockout stage

Take the top 16 teams from the standings table, retrain models (fresh encoder over all historical teams), then play out Round of 16 → Quarterfinals → Semifinals → Final.

Pairings follow standings order (1 vs 2, 3 vs 4, …). Ties in knockout matches are broken at random.

In [8]:
standings_df = pd.read_csv("Data/fifa_worldcup_2026_standings.csv")
qualified_teams = standings_df.head(16)["Team"].tolist()

print(f"\n🏆 Qualified Teams for Knockout Stage: {qualified_teams}")

# Retrain on all 2002+ matches with Kaggle features
df = pd.read_csv("Data/clean_fifa_worldcup_matches.csv").dropna(
    subset=["HomeGoals", "AwayGoals"]
)
if "SameContinent" not in df.columns:
    df = enrich_matches(df, default_kaggle_dir())
df_train = training_matches(df)

team_encoder = LabelEncoder()
team_encoder.fit(pd.concat([df_train["HomeTeam"], df_train["AwayTeam"]]).unique())
df_train = df_train.copy()
df_train["HomeTeamEncoded"] = team_encoder.transform(df_train["HomeTeam"])
df_train["AwayTeamEncoded"] = team_encoder.transform(df_train["AwayTeam"])

X = df_train[FEATURE_COLS]
feature_imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(
    feature_imputer.fit_transform(X), columns=FEATURE_COLS, index=df_train.index
)
y_home = df_train["HomeGoals"]
y_away = df_train["AwayGoals"]

home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)
home_model.fit(X, y_home)
away_model.fit(X, y_away)


def encode_team(team):
    """Return encoded team ID, or -1 if the team never appeared in training data."""
    if team in team_encoder.classes_:
        return team_encoder.transform([team])[0]
    return -1


def knockout_match_features(home_team: str, away_team: str) -> pd.DataFrame:
    """Build imputed feature row for a 2026 knockout pairing (neutral-site style)."""
    row = pd.DataFrame(
        {"Year": [2026], "HomeTeam": [home_team], "AwayTeam": [away_team]}
    )
    row = enrich_matches(row, default_kaggle_dir())
    row["HomeTeamEncoded"] = encode_team(home_team)
    row["AwayTeamEncoded"] = encode_team(away_team)
    return pd.DataFrame(
        feature_imputer.transform(row[FEATURE_COLS]),
        columns=FEATURE_COLS,
    )


🏆 Qualified Teams for Knockout Stage: ['Spain', 'Portugal', 'Switzerland', 'Brazil', 'Germany', 'Belgium', 'England', 'France', 'United States', 'Netherlands', 'Argentina', 'Morocco', 'Iran', 'Uruguay', 'South Korea', 'Turkey']


In [9]:
# Optional sanity check — full 2002+ training set used for knockout models
print(X.shape, y_home.shape, y_away.shape)


(384, 36) (384,) (384,)


In [10]:
knockout_rounds = ["Round of 16", "Quarterfinals", "Semifinals", "Final"]
remaining_teams = qualified_teams.copy()

tournament_results = []  # every knockout match played
final_standings = []     # when each team was eliminated (or won)

for round_name in knockout_rounds:
    print(f"\n🔹 {round_name} Matches:")

    if len(remaining_teams) == 1:
        break

    next_round_teams = []

    # Odd team count: last team in the list gets a bye
    if len(remaining_teams) % 2 == 1:
        bye_team = remaining_teams.pop()
        print(f"⚠️ {bye_team} gets a bye and advances to the next round.")
        next_round_teams.append(bye_team)
        final_standings.append((bye_team, round_name))

    for i in range(0, len(remaining_teams), 2):
        team1, team2 = remaining_teams[i], remaining_teams[i + 1]
        if encode_team(team1) == -1 or encode_team(team2) == -1:
            print(f"⚠️ Error: Team encoding issue with {team1} or {team2}")
            continue

        match_features = knockout_match_features(team1, team2)
        predicted_home_goals = home_model.predict(match_features)[0]
        predicted_away_goals = away_model.predict(match_features)[0]

        if predicted_home_goals > predicted_away_goals:
            winner, loser = team1, team2
        elif predicted_home_goals < predicted_away_goals:
            winner, loser = team2, team1
        else:
            winner = np.random.choice([team1, team2])  # no extra time / penalties modelled
            loser = team1 if winner == team2 else team2

        print(f"⚽ {team1} ({predicted_home_goals:.1f}) vs {team2} ({predicted_away_goals:.1f}) → Winner: {winner}")
        next_round_teams.append(winner)

        tournament_results.append([round_name, team1, team2, predicted_home_goals, predicted_away_goals, winner])
        final_standings.append((loser, round_name))

    remaining_teams = next_round_teams
    print(f"✅ Advancing Teams: {remaining_teams}")

final_winner = remaining_teams[0]
print(f"\n🏆🏆🏆 FIFA World Cup 2026 Champion: {final_winner} 🏆🏆🏆")
final_standings.append((final_winner, "Champion"))

results_df = pd.DataFrame(
    tournament_results,
    columns=["Round", "Team1", "Team2", "Predicted Goals Team1", "Predicted Goals Team2", "Winner"],
)
standings_df = pd.DataFrame(final_standings, columns=["Team", "Position"])

results_df.to_csv("Data/predicted_tournament_results.csv", index=False)
standings_df.to_csv("Data/final_tournament_standings.csv", index=False)

# Append knockout elimination data to the group-stage standings file
previous_standings = pd.read_csv("Data/fifa_worldcup_2026_standings.csv")
updated_standings = pd.concat([previous_standings, standings_df], ignore_index=True)
updated_standings.to_csv("Data/fifa_worldcup_2026_standings_Final_Updated.csv", index=False)



🔹 Round of 16 Matches:
⚽ Spain (1.9) vs Portugal (0.8) → Winner: Spain
⚽ Switzerland (1.2) vs Brazil (1.2) → Winner: Switzerland
⚽ Germany (1.8) vs Belgium (0.9) → Winner: Germany
⚽ England (1.3) vs France (1.2) → Winner: England
⚽ United States (1.0) vs Netherlands (1.9) → Winner: Netherlands
⚽ Argentina (1.6) vs Morocco (1.3) → Winner: Argentina
⚽ Iran (0.8) vs Uruguay (1.5) → Winner: Uruguay
⚽ South Korea (1.0) vs Turkey (1.3) → Winner: Turkey
✅ Advancing Teams: ['Spain', 'Switzerland', 'Germany', 'England', 'Netherlands', 'Argentina', 'Uruguay', 'Turkey']

🔹 Quarterfinals Matches:
⚽ Spain (2.6) vs Switzerland (0.7) → Winner: Spain
⚽ Germany (1.4) vs England (1.4) → Winner: England
⚽ Netherlands (1.8) vs Argentina (1.4) → Winner: Netherlands
⚽ Uruguay (1.7) vs Turkey (1.6) → Winner: Uruguay
✅ Advancing Teams: ['Spain', 'England', 'Netherlands', 'Uruguay']

🔹 Semifinals Matches:
⚽ Spain (1.8) vs England (1.7) → Winner: Spain
⚽ Netherlands (2.3) vs Uruguay (0.8) → Winner: Netherlands